In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

In [3]:
from bayesgpt.simulators import NestedModelFamily, ModelVariant, Tokenizer
from bayesgpt.simulators.benchmarks import SuperDDM, StandardDDM, CollapsingBoundDDM

### Metas

In [4]:
num_samples = 1000  # Global number of samples per model variant

In [5]:
# Define modulation function for context-dependent parameters in SuperDDM
def modulation(params, context):
    """Adjust drift rate based on context (e.g., stimulus strength)."""
    params = params.copy()
    if "v" in params:
        params["v"] = params["v"] * (1 + context[0])  # Scale drift rate
    elif "v_components" in params:
        params["v_components"] = params["v_components"] * (1 + context[0])
    elif "v_schedule" in params:
        params["v_schedule"] = params["v_schedule"] * (1 + context[0])
    return params

In [7]:
# Common tokenizer parameters
parameter_names = [
    "v",
    "a",
    "z",
    "tau",
    "angle",
    "s_v",
    "s_z",
    "s_tau",
]

### DDM Variants

In [7]:
super_ddm_params = parameter_names

In [8]:
free_parameters = {
    "a": lambda c: np.random.uniform(0.8, 1.2, 1),  # Decision boundary
    "s_v": lambda c: np.random.uniform(0.01, 0.1, 1),  # Drift rate noise
    "angle": lambda c: np.random.uniform(0.0, 0.05, 1),  # Boundary collapse
    "s_z": lambda c: np.random.uniform(0.005, 0.02, 1),  # Starting point noise
    "s_tau": lambda c: np.random.uniform(0.005, 0.02, 1),  # Non-decision time noise
    "z": lambda c: np.random.uniform(0.4, 0.6, 1),  # Starting point
    "tau": lambda c: np.random.uniform(0.1, 0.3, 1)  # Non-decision time
}
parameter_dims = {
    "a": 1, "v": 1, "sigma": 1, "s_v": 1, "z": 1, "tau": 1, "angle": 1,
    "s_z": 1, "s_tau": 1,
}

In [9]:
tokenizer_super_mixture = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_super_schedule = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_schedule", "t_schedule", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_standard = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_collapsing = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)

In [10]:
model_variant_mixture = ModelVariant(
    name="super_ddm_mixture",
    model=SuperDDM,
    tokenizer=tokenizer_super_mixture,
    num_samples=num_samples
)
model_variant_schedule = ModelVariant(
    name="super_ddm_schedule",
    model=SuperDDM,
    tokenizer=tokenizer_super_schedule,
    num_samples=num_samples
)
model_variant_standard = ModelVariant(
    name="standard_ddm",
    model=StandardDDM,
    tokenizer=tokenizer_standard,
    num_samples=num_samples
)
model_variant_collapsing = ModelVariant(
    name="collapsing_bound_ddm",
    model=CollapsingBoundDDM,
    tokenizer=tokenizer_collapsing,
    num_samples=num_samples
)

In [11]:
# Cell 3: Run simulations and print results
context = np.array([0.5], dtype=np.float32)  # Single simulation context
result_super_mixture = model_variant_mixture.sample(context=context, num_samples=num_samples)
result_super_schedule = model_variant_schedule.sample(context=context, num_samples=num_samples)
result_standard = model_variant_standard.sample(context=context, num_samples=num_samples)
result_collapsing = model_variant_collapsing.sample(context=context, num_samples=num_samples)

In [12]:
def print_variant_results(variant: ModelVariant, result: dict, tokenizer: Tokenizer, num_samples: int):
    """Print simulation results and summary statistics for a ModelVariant."""
    print(f"\n{variant.name} Results:")
    print("Variant Name:", result["variant_name"])
    print("Simulated Data Keys:", result["sim_data"].keys())
    print("Reaction Times (first 5):", result["sim_data"]["rts"][:5])
    print("Choices (first 5):", result["sim_data"]["choices"][:5])
    print("Full Parameters (shape):", result["full_params"].shape)
    print("Inference Conditions (shape):", result["inference_conditions"].shape)

    # Summarize results using the model's summarize method
    summary = variant.model.summarize(
        outputs=result["sim_data"],
        quantile_levels=[0.1, 0.3, 0.5, 0.7, 0.9],
        by_choice=True,
        tau=np.full(num_samples, result["full_params"][
            tokenizer.parameter_slices["tau"]][0], dtype=np.float32)
    )
    print(f"{variant.name} Summary:")
    print("Invalid Rate:", summary["invalid_rate"])
    print("RT Quantiles:", summary["rt_quantiles"])
    print("RT Quantiles by Choice:\n", summary["rt_quantiles_by_choice"])
    print("Decision Time Quantiles:", summary["dt_quantiles"])
    print("Decision Time Quantiles by Choice:\n", summary["dt_quantiles_by_choice"])

In [13]:
print_variant_results(model_variant_mixture, result_super_mixture, tokenizer_super_mixture, num_samples)
print_variant_results(model_variant_schedule, result_super_schedule, tokenizer_super_schedule, num_samples)
print_variant_results(model_variant_standard, result_standard, tokenizer_standard, num_samples)
print_variant_results(model_variant_collapsing, result_collapsing, tokenizer_collapsing, num_samples)


super_ddm_mixture Results:
Variant Name: super_ddm_mixture
Simulated Data Keys: dict_keys(['rts', 'choices', 'context'])
Reaction Times (first 5): [1.262251  0.7710197 5.000374  0.4994946 5.060596 ]
Choices (first 5): [1. 1. 1. 1. 0.]
Full Parameters (shape): (17,)
Inference Conditions (shape): (34,)
super_ddm_mixture Summary:
Invalid Rate: 0.096
RT Quantiles: [0.85780424 1.4223047  2.3168702  3.5026834  6.38884   ]
RT Quantiles by Choice:
 [[1.744557  2.5463836 3.487353  4.876785  7.27186  ]
 [0.7440882 1.0300204 1.4174742 2.259035  4.99748  ]]
Decision Time Quantiles: [0.6592868 1.2237873 2.1183527 3.3041658 6.190323 ]
Decision Time Quantiles by Choice:
 [[1.5460396  2.347866   3.2888355  4.6782675  7.073343  ]
 [0.54557073 0.8315029  1.2189567  2.0605175  4.7989626 ]]

super_ddm_schedule Results:
Variant Name: super_ddm_schedule
Simulated Data Keys: dict_keys(['rts', 'choices', 'context'])
Reaction Times (first 5): [2.545652   3.069882   3.6336267  0.81662214 1.3036418 ]
Choices (f

### `NestedModelFamily`

In [14]:
# Initialize NestedModelFamily with all variants
model_family = NestedModelFamily(
    variants=[model_variant_mixture, model_variant_schedule, model_variant_standard, model_variant_collapsing],
    n_jobs=2
)

In [15]:
# Batch sample parameters
batch_params = model_family.batch_sample(
    num_samples_per_variant=num_samples,
    context=context
)

In [16]:
print("\nBatch Sampled Parameters:")
for i, params in enumerate(batch_params["parameters"]):
    print(f"Variant {model_family.variant_names[i]} Parameters (first sample):")
    for key, value in params.items():
        print(f"{key}: {value[:5] if isinstance(value, np.ndarray) else value}")


Batch Sampled Parameters:
Variant super_ddm_mixture Parameters (first sample):
v_components: [[-0.29323888 -0.07011601]
 [-0.26160857  0.20223922]
 [-0.3795007   0.45749062]
 [ 0.06119113 -0.24545619]
 [-0.4559151  -0.20093566]]
tau: [[0.2627542 ]
 [0.14453009]
 [0.26543602]
 [0.2801879 ]
 [0.1060578 ]]
s_z: [[0.00798888]
 [0.00502888]
 [0.01718418]
 [0.01825234]
 [0.01125356]]
s_tau: [[0.0056854 ]
 [0.01500895]
 [0.01277327]
 [0.01215856]
 [0.00782575]]
p_components: [[0.6 0.4]
 [0.6 0.4]
 [0.6 0.4]
 [0.6 0.4]
 [0.6 0.4]]
sigma: [[0.05287325]
 [0.12984799]
 [0.07628165]
 [0.05939403]
 [0.08263716]]
s_v: [[0.038259  ]
 [0.04149125]
 [0.04754536]
 [0.01488436]
 [0.04725685]]
z: [[0.47435832]
 [0.43831348]
 [0.5817807 ]
 [0.50405425]
 [0.54980856]]
a: [[1.0206496 ]
 [1.0063449 ]
 [0.95980334]
 [1.1666894 ]
 [1.0798664 ]]
angle: [[0.02442354]
 [0.03899957]
 [0.02962535]
 [0.02681771]
 [0.00927594]]
Variant super_ddm_schedule Parameters (first sample):
v_schedule: [[ 0.5653444   0.6377229

In [17]:
# Run batch simulations with modulation
batch_results = model_family.batch_simulate(
    num_samples_per_variant=num_samples,
    context=[context] * len(model_family.variants),  # Same context for all variants
    modulation=modulation
)

In [18]:
print("\nBatch Simulation Results:")
print(f"sim_data shape: {batch_results['sim_data'].shape}")  # (4, 1000, 2)
print(f"full_params shape: {batch_results['full_params'].shape}")  # (4, 17)
print(f"inference_conditions shape: {batch_results['inference_conditions'].shape}")  # (4, 34)
print(f"sampled_parameters shape: {batch_results['sampled_parameters'].shape}")  # (4,)
print(f"variant_names: {batch_results['variant_names']}")
print(f"variant_indices shape: {batch_results['variant_indices'].shape}")  # (4, 1)
print(f"context shape: {batch_results['context'].shape}")  # (4, 1)


Batch Simulation Results:
sim_data shape: (4, 1000, 2)
full_params shape: (4, 17)
inference_conditions shape: (4, 34)
sampled_parameters shape: (4,)
variant_names: ['super_ddm_mixture', 'super_ddm_schedule', 'standard_ddm', 'collapsing_bound_ddm']
variant_indices shape: (4, 1)
context shape: (4, 1)


In [19]:
for i, params in enumerate(batch_results["sampled_parameters"]):
    print(f"\nVariant {batch_results['variant_names'][i]} Sampled Parameters (first 5 samples):")
    for key, value in params.items():
        print(f"{key}: {value[:5] if isinstance(value, np.ndarray) else value}")


Variant super_ddm_mixture Sampled Parameters (first 5 samples):
v_components: [[ 0.47555327  0.09774452]
 [ 0.02283196 -0.15778333]
 [ 0.60582787  0.00770678]
 [ 0.64975625  0.21467058]
 [-0.20339681 -0.34353173]]
tau: [[0.22919324]
 [0.11011978]
 [0.1817969 ]
 [0.1070013 ]
 [0.13255827]]
s_tau: [[0.00988524]
 [0.00576071]
 [0.00752231]
 [0.00947237]
 [0.00877624]]
angle: [[0.03409381]
 [0.0444374 ]
 [0.01821876]
 [0.0177977 ]
 [0.04221193]]
sigma: [[0.10300309]
 [0.11266477]
 [0.13900255]
 [0.08239097]
 [0.11040741]]
s_v: [[0.04030986]
 [0.02557798]
 [0.08224219]
 [0.06465553]
 [0.09017479]]
z: [[0.4960871 ]
 [0.47895646]
 [0.509815  ]
 [0.5437119 ]
 [0.43563303]]
a: [[1.1130601]
 [1.0481377]
 [0.8332606]
 [1.1408838]
 [0.9444491]]
s_z: [[0.00647326]
 [0.01567341]
 [0.01479159]
 [0.01079179]
 [0.01046256]]

Variant super_ddm_schedule Sampled Parameters (first 5 samples):
v_schedule: [[ 0.63586134 -0.33301198]
 [-0.34204566 -0.34609768]
 [ 0.21773164  0.13154262]
 [ 0.14424261  0.7479